In [ ]:
# comet-ml must be imported before torch and sklearn
import comet_ml
import scanpy as sc
import umap
import numpy as np
import torch
import scanpy as sc
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score

import scarches as sca
random_seed = 42
sca.models.mvTCR.utils_training.fix_seeds(random_seed)

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [15]:
# Load
adata = sc.read_h5ad('merged_EAE.h5ad')
adata

AnnData object with n_obs × n_vars = 66525 × 2500
    obs: 'tissue', 'mouse_id', 'receptor_type', 'receptor_subtype', 'chain_pairing', 'clonotype', 'clonotype_size', 'alpha_len', 'beta_len'
    var: 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection'
    uns: 'aa_to_id', 'chain_indices', 'clonotype', 'ir_dist_nt_identity'
    obsm: 'airr', 'alpha_seq', 'beta_seq', 'chain_indices'

In [16]:
adata.obs['set'] = adata.obs['mouse_id'].astype(str).apply(lambda x: 'test' if x.startswith('b') else 'train')

train_mask = adata.obs['set'] == 'train'
train_indices = adata.obs.index[train_mask]
n_val = int(0.2 * len(train_indices))
if n_val > 0:
    val_indices = np.random.choice(train_indices, size=n_val, replace=False)
    adata.obs.loc[val_indices, 'set'] = 'val'

In [21]:
# import mvtcr.utils_training as utils
model = utils.load_model(adata, './saved_models'+ '/merged/DE2500_epoch50' +'/trial_0/best_model_by_metric.pt')
